# Lab 8 · Notebook 1: Quicker figures with pandas and seaborn

**Time:** about 40 minutes. **What you will do:** draw the Lab 7 figures again in fewer lines. First pandas, which can plot a column or a table directly. Then seaborn, where you name the columns and it does the rest.

Both are built on matplotlib. So everything you learned in Lab 7 still works: you make `fig, ax = plt.subplots()`, pass `ax=ax`, and label `ax` as before.

Every **Write it** cell has a self-check underneath. It looks at `ax` (or `fig`) to see what you drew, so keep those names.

The table is the lecture's DepMap file: 1,165 cell lines and 257 genes. A gene effect near 0 means the cells grow fine without the gene. Around -1 or below means they die without it.

In [ ]:
# Run this first. It loads the DepMap table from the lecture.
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab08/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE).copy()    # .copy() packs the 261 columns together, so adding a column later is quick
skin = df[df["oncotree_lineage"] == "Skin"]
print("cell lines:", len(df), "  genes:", df.shape[1] - 4, "  skin lines:", len(skin))

## 1. pandas: the column plots itself

A pandas Series has a `.plot` attribute. `.plot.hist()` draws a histogram, `.plot.barh()` horizontal bars, `.plot.scatter()` a scatter plot. Pass `ax=ax` so it draws on your panel.

Compare with Lab 7, where you wrote `ax.hist(df["SOX10"], bins=40)`. The result is the same; only who calls whom has changed.

In [ ]:
# DEMO - pandas draws the histogram
fig, ax = plt.subplots()
df["SOX10"].plot.hist(bins=40, ax=ax)
ax.set_xlabel("SOX10 gene effect")
plt.show()

### Write it

**Your task:** use pandas to draw a histogram of `BRAF` across all cell lines with **30** bins. Label the x axis.

In [ ]:
# TODO - a pandas histogram of BRAF



# --- self-check (do not edit) ---
assert len(ax.patches) == 30, f"Your histogram has {len(ax.patches)} bars. Use bins=30"
total = sum(bar.get_height() for bar in ax.patches)
assert total == 1165, f"Your bars add up to {total:.0f} cell lines, not 1,165. Use the whole table, df"
assert abs(ax.patches[0].get_x() - df["BRAF"].min()) < 1e-9, "The bars do not start at the lowest BRAF value. Did you plot the BRAF column?"
assert "BRAF" in ax.get_xlabel(), "Label the x axis with the gene name"
print("Correct: 30 bins, all 1,165 cell lines")

## 2. A table result plots itself too

This is where pandas plotting shines. A `groupby` gives you a Series with one value per group, and that Series can draw itself straight away.

Below: the median SOX10 effect in every lineage, the 8 lowest kept, then drawn as horizontal bars. `fig.tight_layout()` makes room for the long lineage names.

In [ ]:
# DEMO - median SOX10 per lineage, the 8 lowest
medians = df.groupby("oncotree_lineage")["SOX10"].median()
lowest = medians.nsmallest(8)
print(lowest.round(2))

fig, ax = plt.subplots()
lowest.plot.barh(ax=ax)
ax.set_xlabel("median SOX10 gene effect")
fig.tight_layout()
plt.show()

### Write it

**Predict first:** BRAF is the gene mutated in about half of all melanomas. Which lineage will have the lowest median BRAF effect?

**Your task:** draw the same kind of figure for `BRAF`: the median per lineage, the **10** lowest, as horizontal bars. Store the 10 values in `lowest_braf`. Label the x axis.

In [ ]:
# TODO - median BRAF per lineage, the 10 lowest



# --- self-check (do not edit) ---
expected = df.groupby("oncotree_lineage")["BRAF"].median().nsmallest(10)
assert len(lowest_braf) == 10, f"lowest_braf has {len(lowest_braf)} values. Use .nsmallest(10)"
assert lowest_braf.round(6).equals(expected.round(6)), "lowest_braf does not match. Group by oncotree_lineage, take the BRAF median, then .nsmallest(10)"
assert len(ax.patches) == 10, f"Your panel has {len(ax.patches)} bars. Plot lowest_braf with .plot.barh(ax=ax)"
assert ax.get_xlabel() != "", "Label the x axis"
print("Correct:", lowest_braf.index[0], "has the lowest median BRAF effect")

pandas is the quickest way to **look**: one line after a `groupby`, no loop, no lists of labels. When you want a figure that compares groups and shows every data point, seaborn is the better tool.

## 3. seaborn: name the columns

seaborn works on a whole table. You give it `data=` (the DataFrame) and say which **column** goes where: `x=`, `y=`. It finds the numbers itself.

```python
sns.histplot(data=df, x="SOX10", ax=ax)
sns.scatterplot(data=df, x="BRAF", y="SOX10", ax=ax)
```

seaborn also labels the axes with the column names, so you only change them when you want something clearer.

In [ ]:
# DEMO - the SOX10 histogram, seaborn style
fig, ax = plt.subplots()
sns.histplot(data=df, x="SOX10", bins=40, ax=ax)
plt.show()

### Write it

**Your task:** with seaborn, draw a scatter plot of the **skin** lines only: `BRAF` on x, `SOX10` on y. Use `sns.scatterplot(data=..., x=..., y=..., ax=ax)`.

In [ ]:
# TODO - seaborn scatter, skin only



# --- self-check (do not edit) ---
import numpy as np
assert len(ax.collections) == 1, f"Your panel has {len(ax.collections)} scatter layers. Call sns.scatterplot once"
points = ax.collections[0].get_offsets()
assert len(points) == 73, f"Your scatter has {len(points)} dots. Use data=skin, the 73 skin lines"
assert np.allclose(points[:, 0], skin["BRAF"]), "The x values are not BRAF. Use x=\"BRAF\""
assert np.allclose(points[:, 1], skin["SOX10"]), "The y values are not SOX10. Use y=\"SOX10\""
print("Correct: 73 skin lines, and seaborn labelled the axes for you:", ax.get_xlabel(), "and", ax.get_ylabel())

## 4. Groups need a label column

To compare groups, seaborn wants **one column that says which group each row belongs to**. Our table does not have one yet, so we make it. The pattern from the lecture:

```python
df["group"] = "everything else"                # 1. every row gets the default label
df.loc[is_skin, "group"] = "skin"              # 2. relabel only the rows where is_skin is True
```

Line 1 really does put the same text into every row. That is the point: it is the starting label. Line 2 overwrites it, but only in the rows you select with `.loc[rows, column]`.

### Broken

This cell is meant to label the 73 skin lines `"skin"` and everything else `"everything else"`. Run it and read the self-check. Then fix the second line.

In [ ]:
# BROKEN - only the skin rows should say "skin"
is_skin = df["oncotree_lineage"] == "Skin"
df["group"] = "everything else"
df["group"] = "skin"


# --- self-check (do not edit) ---
counts = df["group"].value_counts()
print(counts)
assert counts.get("skin", 0) != 1165, "Every row says skin. df[\"group\"] = ... sets the whole column. Use df.loc[is_skin, \"group\"] = \"skin\""
assert counts.get("skin", 0) == 73, f"{counts.get('skin', 0)} rows say skin. There are 73 skin lines"
assert counts.get("everything else", 0) == 1092, "The other 1,092 rows should still say everything else"
print("Fixed: 73 skin, 1,092 everything else")

## 5. Box plot with every cell line shown

With a group column, the figure from Lab 7 takes two calls. The box summarises each group; the strip plot draws every cell line as a dot on top. `order=` fixes the left-to-right order.

This demo uses three groups: melanoma, the other skin lines, and everything else. Melanomas are found by their disease name. Uveal melanomas in the eye count too, which is why the melanoma group is 74 lines.

In [ ]:
# DEMO - three groups, box plus dots
is_skin = df["oncotree_lineage"] == "Skin"
is_melanoma = df["oncotree_primary_disease"].str.contains("Melanoma")

df["group"] = "everything else"
df.loc[is_skin, "group"] = "other skin"
df.loc[is_melanoma, "group"] = "melanoma"
print(df["group"].value_counts())

order = ["melanoma", "other skin", "everything else"]
fig, ax = plt.subplots()
sns.boxplot(data=df, x="group", y="SOX10", order=order, ax=ax, showfliers=False)
sns.stripplot(data=df, x="group", y="SOX10", order=order, ax=ax, color="grey", size=2)
ax.set_xlabel("")
plt.show()

`showfliers=False` stops the box plot drawing its own outlier dots, because the strip plot already shows every point.

### Write it

**Predict first:** MDM2 switches off the tumour suppressor p53. Will melanoma need MDM2 more than everything else, or less?

**Your task:** draw the same figure for `MDM2`: a box plot plus a grey strip plot, same three groups in the same order. Add the title `MDM2 by group`.

In [ ]:
# TODO - MDM2: box plus dots, three groups



# --- self-check (do not edit) ---
labels = [t.get_text() for t in ax.get_xticklabels()]
assert labels == order, f"The groups read {labels}. Pass order=order to both calls"
dots = sum(len(c.get_offsets()) for c in ax.collections)
assert dots == 1165, f"The panel shows {dots} dots. Add sns.stripplot(..., ax=ax) so every cell line is shown"
assert ax.get_ylabel() == "MDM2", f"The y axis is {ax.get_ylabel()!r}. Use y=\"MDM2\" in both calls"
assert ax.get_title() == "MDM2 by group", "Add the title MDM2 by group"
print("Correct: three boxes and all 1,165 cell lines")

## 6. A function that returns a figure

Your project asks for a function that **builds a figure and returns it**, instead of only showing it. Returning `fig` means whoever calls the function can save it, or test it.

```python
def group_figure(table, gene):
    fig, ax = plt.subplots()
    ...                      # draw on ax
    return fig
```

The function should only use what it is given: `table` and `gene`. Inside it, write `data=table` and `y=gene`, not `df` and `"MDM2"`.

### Write it

**Your task:** write `group_figure(table, gene)`. It draws the box-plus-dots figure from section 5 for any gene, sets the title to the gene name, and returns `fig`. It can assume `table` already has the `"group"` column.

In [ ]:
# TODO - a function that returns the three-group figure
def group_figure(table, gene):



# --- self-check (do not edit) ---
from matplotlib.figure import Figure
test = group_figure(df, "TP53")
assert isinstance(test, Figure), f"group_figure returned {type(test).__name__}. End the function with return fig"
test_ax = test.axes[0]
assert test_ax.get_title() == "TP53", f"The title is {test_ax.get_title()!r}. Use ax.set_title(gene)"
assert test_ax.get_ylabel() == "TP53", "The y axis is not TP53. Use y=gene, not a fixed gene name"
assert sum(len(c.get_offsets()) for c in test_ax.collections) == 1165, "Not every cell line is shown. Add the strip plot"
small = group_figure(df.head(100), "SOX10")
assert sum(len(c.get_offsets()) for c in small.axes[0].collections) == 100, "Given 100 rows, your figure shows a different number of dots. Use data=table, not df"
plt.close("all")
print("Correct: group_figure works for any table and any gene")

Now use it. The function returns the figure, so you can save it with `fig.savefig(...)`. Try a few genes: `"TP53"`, `"BRAF"`, `"KRAS"`, `"RAN"`.

In [ ]:
# DEMO - call your function, look at it, save it
fig = group_figure(df, "TP53")
fig.savefig("tp53_by_group.png", dpi=300, bbox_inches="tight")
plt.show()

Melanoma lines grow **better** without TP53 (values above 0) and die without MDM2. Together that says p53 is still working in most melanomas, held in check by MDM2. Notebook 2 asks whether these differences are bigger than chance.

## If you finish early

1. `sns.violinplot` takes the same arguments as `sns.boxplot`. Swap it into `group_figure` and compare. Which shape tells you more about the 8 "other skin" lines?
2. pandas can draw a box plot per group too: `df.boxplot(column="SOX10", by="group", ax=ax)`. Is it quicker? Is it as clear?
3. Browse the [seaborn gallery](https://seaborn.pydata.org/examples/index.html) and pick one figure type that would suit your project data.